# v02 — Few-shot

End-to-end Colab runner for `v02_fewshot`. All knobs (model, dtype, batch size, max tokens, ...) live in [`app/physics_solution/config.py`](../../config.py) — edit there, not here.

Runs **directly off the Drive copy** (Colab Oct-2025+ runtime is fast enough that copying to `/content/` is no longer worth the disk churn). The repo path `REPO_ROOT` below points to your Drive folder.

## 1. Mount Drive + chdir


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_ROOT = '/content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils'  # change if needed
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

Mounted at /content/drive
cwd: /content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils


In [2]:
!pip -q install -r app/physics_solution/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 162.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 156.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.9.0 which is incompatible.


### Install Qwen3.5 fast-attention kernels (`flash-linear-attention` + `causal-conv1d`)

On Colab Oct-2025+ runtimes both wheels install cleanly. If `causal_conv1d: False` shows up, the model still runs via torch fallback at ~3× slower — accuracy is unaffected.

In [3]:
from app.physics_solution.shared.colab.setup import install_fast_kernels
install_fast_kernels()

[install_fast_kernels] env: python=cp312 torch=2.8 cuda=12.6 -> tag cu12
[install_fast_kernels] flash-linear-attention ...
  fla: OK
[install_fast_kernels] causal-conv1d (pre-built wheels) ...
  trying v1.5.2 abi=FALSE ...
  causal-conv1d: OK (1.5.2, abi=FALSE)


{'fla': True, 'causal_conv1d': True}

In [4]:
# setup_colab handles .env + LangSmith.
from app.physics_solution.shared.colab.setup import setup_colab
setup_colab(repo_root=REPO_ROOT)

[colab_setup] loaded /content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils/app/physics_solution/.env
               cwd: /content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils
          hf_token: OK
         langsmith: OFF
        base_model: Qwen/Qwen3.5-4B
             dtype: bf16
            device: cuda
        batch_size: 100
    max_new_tokens: 640
       temperature: 0.0
         test_file: app/physics_solution/data/test/full_test.csv


{'cwd': '/content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils',
 'hf_token': 'OK',
 'langsmith': 'OFF',
 'base_model': 'Qwen/Qwen3.5-4B',
 'dtype': 'bf16',
 'device': 'cuda',
 'batch_size': 100,
 'max_new_tokens': 640,
 'temperature': 0.0,
 'test_file': 'app/physics_solution/data/test/full_test.csv'}

In [5]:
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-7f035d0e-eaf1-b7a7-9f15-f28c1f2256ba)


## 2. Test set paths

Two evaluation runs: **full test** (original data) and **golden** (DeepSeek-rewritten CoT).

| File | Rows | Scope |
|---|---|---|
| `full_test.csv` | 1352 | All answer types — original CoT |
| `deepseek-v4-pro_golden_data.csv` | 1352 | All answer types — golden CoT |

In [6]:
import os

# --- Full test (original data) ---
FULL_TEST_FILE  = 'app/physics_solution/data/test/full_test.csv'
FULL_OUT_FILE   = 'app/physics_solution/versions/v02_fewshot/output/results_full.json'

# --- Golden test (DeepSeek-rewritten CoT) ---
GOLDEN_TEST_FILE = 'app/physics_solution/data/golden/deepseek-v4-pro_golden_data.csv'
GOLDEN_OUT_FILE  = 'app/physics_solution/versions/v02_fewshot/output/results_golden.json'

for label, f in [('Full', FULL_TEST_FILE), ('Golden', GOLDEN_TEST_FILE)]:
    assert os.path.exists(f), f"Not found: {f}"
    print(f'{label}: {f}')

Full: app/physics_solution/data/test/full_test.csv
Golden: app/physics_solution/data/golden/deepseek-v4-pro_golden_data.csv


## 3. Curate few-shot examples

Picks K examples per domain prefix (drops anything in `sample_test.csv` to avoid leakage). With the full 973-sample test, the few-shot pool is tiny — re-curate with --k 1 if pool too small.

In [7]:
!python -m app.physics_solution.versions.v02_fewshot.select_fewshot --k 3 --seed 42

Excluding 50 test IDs
Candidate pool: 1302 rows (all answer types) across 8 domains
  sci_notation candidates: 365
Wrote 24 examples to /content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils/app/physics_solution/versions/v02_fewshot/input/examples.json
Breakdown: {'CH': 3, 'CHLT': 3, 'DDT': 3, 'DT': 3, 'LD': 3, 'NL': 3, 'TD': 3, 'THCB': 3}


## 4. Inference

Runs both full test and golden test sequentially. All knobs (batch size, dtype, max_new_tokens, ...) come from `config.py`.

In [10]:
# --- 4a. Full test ---
!python -m app.physics_solution.cli.inference \
    --version v02_fewshot \
    --test-file {FULL_TEST_FILE} \
    --out {FULL_OUT_FILE} \
    --n-examples 3 \
    --batch-size 80

[v02_fewshot] loading Qwen/Qwen3.5-4B dtype=bf16 device=cuda batch_size=80 prefix=''
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100% 2/2 [00:00<00:00, 36631.48it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 426/426 [00:00<00:00, 6438.66it/s]
Running on 1352 questions (batch_size=80)
  0% 0/1352 [00:00<?, ?it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
  6% 80/1352 [01:08<18:16,  1.16it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 12% 160/1352 [02:19<17:18,  1.15it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 18% 240/1352 [03:24<15:43,  1.18it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 24% 320/1352 [04:17<13:15,  1.30it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 30% 400/1352 [05:01<11:01,  1.44it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 36% 480/1352 [06:11<10:

In [11]:
# --- 4b. Golden test ---
!python -m app.physics_solution.cli.inference \
    --version v02_fewshot \
    --test-file {GOLDEN_TEST_FILE} \
    --out {GOLDEN_OUT_FILE} \
    --n-examples 3 \
    --batch-size 80

[v02_fewshot] loading Qwen/Qwen3.5-4B dtype=bf16 device=cuda batch_size=80 prefix=''
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100% 2/2 [00:00<00:00, 30504.03it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 426/426 [00:00<00:00, 5661.71it/s]
Running on 1352 questions (batch_size=80)
  0% 0/1352 [00:00<?, ?it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
  6% 80/1352 [00:51<13:31,  1.57it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 12% 160/1352 [01:48<13:34,  1.46it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 18% 240/1352 [02:54<13:50,  1.34it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 24% 320/1352 [03:46<12:13,  1.41it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 30% 400/1352 [04:31<10:25,  1.52it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 36% 480/1352 [05:27<09:

## 5. Push to HF (with full metadata)


In [12]:
!python -m app.physics_solution.cli.upload_model \
    --version-num 2 --strategy fewshot \
    --results {FULL_OUT_FILE} \
    --test-file {FULL_TEST_FILE} \
    --extra-results {GOLDEN_OUT_FILE}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 14 files:   0% 0/14 [00:00<?, ?it/s]
Fetching 14 files:   7% 1/14 [00:02<00:30,  2.32s/it]
Fetching 14 files:  21% 3/14 [00:03<00:10,  1.04it/s]
Fetching 14 files: 100% 14/14 [00:04<00:00,  3.46it/s]
Download complete: 100% 77.7k/77.7k [00:04<00:00, 21.7kB/s]                Snapshotted Qwen/Qwen3.5-4B -> /content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils/.hf_snapshots/Qwen__Qwen3.5-4B
Download complete: 100% 77.7k/77.7k [00:04<00:00, 17.0kB/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...Qwen3.5-4B/tokenizer.json: 100% 12.8M/12.8M [00:00<?, ?B/s]

Process

## 6. Inspect results


In [13]:
import json, pandas as pd

for label, path in [('FULL', FULL_OUT_FILE), ('GOLDEN', GOLDEN_OUT_FILE)]:
    print(f'\n{"="*60}')
    print(f'  {label}: {path}')
    print(f'{"="*60}')
    data = json.loads(open(path, encoding='utf-8').read())
    print('summary:', json.dumps(data['summary'], indent=2, ensure_ascii=False))
    df = pd.DataFrame(data['rows'])
    wrong = df[~df['is_correct']].copy()
    wrong['reached_final'] = wrong['completion'].str.contains('FINAL ANSWER', case=False, regex=False)
    print(f"\nWrong: {len(wrong)} / {len(df)}")
    print(f"  ... never reached FINAL ANSWER: {(~wrong['reached_final']).sum()}")
    display(wrong[['id', 'gold_answer', 'pred_numeric', 'pred_unit', 'gold_unit', 'reached_final']].head(20))


  FULL: app/physics_solution/versions/v02_fewshot/output/results_full.json
summary: {
  "n": 1352,
  "correct": 700,
  "accuracy": 0.5177514792899408,
  "mean_latency_s": 0.6245920342453838,
  "version": "v02_fewshot",
  "model_id": "Qwen/Qwen3.5-4B",
  "dtype": "bf16",
  "effective_dtype": "bf16",
  "batch_size": 80,
  "assistant_prefix": "",
  "description": "Few-shot prompting: K worked examples per domain are injected as prior user/assistant turns before the real question. Examples are curated by `select_fewshot.py` from the training data (test-set IDs excluded).",
  "n_examples": 3
}

Wrong: 652 / 1352
  ... never reached FINAL ANSWER: 5


,id,gold_answer,pred_numeric,pred_unit,gold_unit,reached_final
0,LD343,2.027*10^6,1.944000e+07,N/C,V/m,True
1,LD248,0.495,4.950000e-02,N,N,True
2,LD110,0.023,4.423000e+00,N,N,True
3,LD151,5.09*10^-4,5.091000e+00,N,N,True
6,LD274,36.32,2.117600e+00,N,N,True
7,LD377,8.44 × 10^6,1.040000e+07,V/m,V/m,True
9,LD049,14.34,1.510000e+00,N,N,True
12,LD098,1.152 × 10^7,8.640000e+05,N/C,V/m,True
13,LD087,-2\sqrt{2} x q,0.000000e+00,C,-,True
15,LD385,3.50 × 10^6,2.830000e+06,V/m,V/m,True



  GOLDEN: app/physics_solution/versions/v02_fewshot/output/results_golden.json
summary: {
  "n": 1352,
  "correct": 707,
  "accuracy": 0.5229289940828402,
  "mean_latency_s": 0.5435785349656844,
  "version": "v02_fewshot",
  "model_id": "Qwen/Qwen3.5-4B",
  "dtype": "bf16",
  "effective_dtype": "bf16",
  "batch_size": 80,
  "assistant_prefix": "",
  "description": "Few-shot prompting: K worked examples per domain are injected as prior user/assistant turns before the real question. Examples are curated by `select_fewshot.py` from the training data (test-set IDs excluded).",
  "n_examples": 3
}

Wrong: 645 / 1352
  ... never reached FINAL ANSWER: 5


,id,gold_answer,pred_numeric,pred_unit,gold_unit,reached_final
0,LD343,2.027 * 10^{6},1.944000e+07,N/C,V/m,True
1,LD248,0.495,4.950000e-02,N,N,True
2,LD110,0.023,4.423000e+00,N,N,True
3,LD151,5.09 * 10^{-4},5.091000e+00,N,N,True
6,LD274,36.32,2.117600e+00,N,N,True
7,LD377,8.44 * 10^{6},1.040000e+07,V/m,V/m,True
9,LD049,14.34,1.510000e+00,N,N,True
12,LD098,1.152 * 10^{7},8.640000e+05,N/C,V/m,True
13,LD087,-2\sqrt{2} q,0.000000e+00,C,-,True
15,LD385,3.50 * 10^{6},2.830000e+06,V/m,V/m,True


## 7. Deep error analysis

Open [`app/physics_solution/eda/notebooks/error_analysis.ipynb`](../../eda/notebooks/error_analysis.ipynb) — it categorises every wrong row into fail modes (format / order-of-magnitude / unit / domain-specific / suspected-gold-error) and writes a markdown report.